# ATLAS — PDF to JSON Converter
**Arquiteto de Transformação e Limpeza de Assets Científicos**

> Este notebook converte PDFs científicos em JSONs estruturados, limpos e parseáveis para consumo por agentes de análise e pipelines RAG.

## Arquitetura (DRY)
| Módulo | Responsabilidade |
|---|---|
| **Célula 1** | Dependências |
| **Célula 2** | Configuração + Resolução de caminhos |
| **Célula 3** | Extratores de texto (pdfplumber / pymupdf) |
| **Célula 4** | Parsers de estrutura (título, abstract, seções, refs) |
| **Célula 5** | Motor central: `convert_pdf()` + `process_folder()` |
| **Célula 6** | Execução interativa (melhor via `run_all()`) |

---
⚠️ **Idempotente**: PDFs já convertidos são automaticamente pulados.  
🔒 **Antifragil**: falhas isoladas por arquivo nunca abortam o lote inteiro.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1 — DEPENDÊNCIAS
# Execute uma vez. Seguro repetir: já instalados são detectados.
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

REQUIRED = {
    'pdfplumber': 'pdfplumber',  # importname: package_name
    'fitz':       'pymupdf',
}

for import_name, pkg_name in REQUIRED.items():
    try:
        __import__(import_name)
        print(f'✅ {pkg_name} já instalado')
    except ImportError:
        print(f'📦 Instalando {pkg_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name, '-q'])
        print(f'✅ {pkg_name} instalado')

print('\n🚀 Dependências prontas!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2 — CONFIGURAÇÃO E RESOLUÇÃO DE CAMINHOS
# Antifragil: resolve REPO_ROOT por múltiplas estratégias.
# ═══════════════════════════════════════════════════════════════
from pathlib import Path
import os, sys

# ── Resolução de REPO_ROOT (3 estratégias em cascata) ──────────
def _find_repo_root() -> Path:
    """Localiza a raiz do repositório por marcadores conhecidos."""
    anchor_files = {'.git', 'picoc-method', 'src', '.agents'}

    # Estratégia 1: subir da CWD até encontrar todos os marcadores
    candidate = Path(os.getcwd())
    for _ in range(10):
        present = {f.name for f in candidate.iterdir()}
        if anchor_files.issubset(present):
            return candidate
        parent = candidate.parent
        if parent == candidate:
            break
        candidate = parent

    # Estratégia 2: subir do arquivo do notebook (__file__ pode não existir em Jupyter)
    try:
        nb_path = Path(globals().get('__vsc_ipynb_file__') or __file__)
        candidate = nb_path.resolve()
        for _ in range(10):
            present = {f.name for f in candidate.iterdir()}
            if anchor_files.issubset(present):
                return candidate
            candidate = candidate.parent
    except Exception:
        pass

    # Estratégia 3: fallback manual — edite apenas se as duas anteriores falharem
    fallback = Path(r'c:\Users\evers\Desktop\corporative-rag-research-picoc')
    if fallback.exists():
        return fallback

    raise RuntimeError(
        'Não foi possível localizar REPO_ROOT automaticamente.\n'
        'Edite a variável REPO_ROOT manualmente nesta célula.'
    )

REPO_ROOT = _find_repo_root()

# ── Caminhos derivados (Single Source of Truth) ─────────────────
ARTICLES_ROOT = REPO_ROOT / 'picoc-method' / 'material' / 'articles'
OUTPUT_ROOT   = REPO_ROOT / 'src' / 'scripts_outputs' / 'articles-outputs'

# Catálogo de pastas conhecidas (extenda aqui se surgir nova pasta)
FOLDER_CATALOG: dict[str, Path] = {
    'best-sources':         ARTICLES_ROOT / 'best-sources',
    'all-sources-filtered': ARTICLES_ROOT / 'all-sources-filtered',
    'alt-sources':          ARTICLES_ROOT / 'alt-sources',
}

CONVERTER_VERSION = '2.0'

# ── Garantir diretórios de output ───────────────────────────────
for name in FOLDER_CATALOG:
    (OUTPUT_ROOT / name).mkdir(parents=True, exist_ok=True)

# ── Diagnóstico de ambiente ─────────────────────────────────────
print(f'📁 REPO_ROOT : {REPO_ROOT}')
print(f'📂 ARTIGOS   : {ARTICLES_ROOT}')
print(f'📤 OUTPUT    : {OUTPUT_ROOT}')
print()
for name, path in FOLDER_CATALOG.items():
    pdfs = sorted(path.glob('*.pdf')) if path.exists() else []
    jsons = sorted((OUTPUT_ROOT / name).glob('*.json')) if (OUTPUT_ROOT / name).exists() else []
    pending = len(pdfs) - len([j for j in jsons if (path / (j.stem + '.pdf')).exists()])
    icon = '✅' if path.exists() else '❌'
    print(f'  {icon} {name}: {len(pdfs)} PDFs | {len(jsons)} JSONs existentes | ~{max(0,pending)} pendentes')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3 — EXTRATORES DE TEXTO
# Dois backends independentes com contrato idêntico:
#   -> (pages: list[str], n_pages: int, warnings: list[str])
# Nunca lançam exceção: falhas viram warnings.
# ═══════════════════════════════════════════════════════════════
from pathlib import Path

ExtractionResult = tuple[list[str], int, list[str]]  # pages, n_pages, warnings


def _extract_pdfplumber(pdf_path: Path) -> ExtractionResult:
    """Backend primário: pdfplumber. Robusto para texto selecionável."""
    try:
        import pdfplumber
        pages, warnings = [], []
        with pdfplumber.open(pdf_path) as pdf:
            n = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                try:
                    text = page.extract_text() or ''
                    if text.strip():
                        pages.append(text)
                    else:
                        warnings.append(f'p{i+1}: sem texto extraível')
                except Exception as e:
                    warnings.append(f'p{i+1}: erro de extração — {e}')
        return pages, n, warnings
    except Exception as e:
        return [], 0, [f'pdfplumber falhou globalmente: {e}']


def _extract_pymupdf(pdf_path: Path) -> ExtractionResult:
    """Backend fallback: pymupdf. Mais tolerante a PDFs malformados."""
    try:
        import fitz  # pymupdf
        pages, warnings = [], []
        doc = fitz.open(str(pdf_path))
        n = len(doc)
        for i, page in enumerate(doc):
            try:
                text = page.get_text() or ''
                if text.strip():
                    pages.append(text)
                else:
                    warnings.append(f'p{i+1}: vazia ou baseada em imagem')
            except Exception as e:
                warnings.append(f'p{i+1}: erro — {e}')
        doc.close()
        return pages, n, warnings
    except Exception as e:
        return [], 0, [f'pymupdf falhou globalmente: {e}']


def extract_text(pdf_path: Path) -> tuple[str, int, str, list[str]]:
    """
    Interface única de extração. Tenta pdfplumber, faz fallback para pymupdf.
    Retorna: (full_text, n_pages, tool_used, warnings)
    NUNCA levanta exceção — falhas críticas retornam texto vazio + warnings.
    """
    pages, n_pages, warnings = _extract_pdfplumber(pdf_path)
    tool = 'pdfplumber'

    if not pages:
        warnings.append('Activating fallback: pymupdf')
        pages, n_pages, fb_warnings = _extract_pymupdf(pdf_path)
        warnings.extend(fb_warnings)
        tool = 'pymupdf'

    if not pages:
        warnings.append('CRÍTICO: nenhum texto extraível. PDF pode ser imagem/scan.')

    return '\n\n'.join(pages), n_pages, tool, warnings


print('✅ Extratores carregados (pdfplumber + pymupdf).')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4 — PARSERS DE ESTRUTURA
# Cada parser é independente, puro (sem side-effects) e defensivo.
# Campos não encontrados retornam None/[] — NUNCA valores inventados.
# ═══════════════════════════════════════════════════════════════
import re
from typing import Optional

# ── Padrões compilados (compilados uma vez, reutilizados N vezes) ──
_RE_YEAR       = re.compile(r'\b(20[12]\d)\b')
_RE_DOI        = re.compile(r'\b(10\.\d{4,}/[^\s]+)', re.IGNORECASE)
_RE_ABSTRACT   = re.compile(
    r'(?:Abstract|ABSTRACT)[:\s—\-]+(.{80,3000}?)(?=\n\n|\n[A-Z1-9]|\Z)',
    re.DOTALL
)
_RE_KEYWORDS   = re.compile(
    r'(?:Keywords|KEYWORDS|Index Terms)[:\s—]+([^\n]{10,400})',
    re.IGNORECASE
)
_RE_SECTION    = re.compile(
    r'^(?:'                           # início de linha
    r'\d{1,2}[\.]?\s+[A-Z][^\n]{3,70}'  # "1. Introduction" / "1 Introduction"
    r'|[A-ZÁÉÍÓÚ][A-ZÁÉÍÓÚ\s]{4,60}'   # "INTRODUCTION"
    r'|(?:Abstract|Introduction|Conclusion|Related Work|References|Methodology)[:\s]'
    r')$'
)
_RE_REF_SPLIT  = re.compile(r'\n\[\d+\]|\n\d+\.\s|\n\d+\s')
_RE_REF_SECTION = re.compile(
    r'(?:References|Bibliography|REFERENCES|BIBLIOGRAPHY)[\s\n]+(.+?)$',
    re.DOTALL | re.IGNORECASE
)


def parse_title(text: str) -> Optional[str]:
    """
    Heurística conservadora: primeira linha não-numérica com 10-250 chars
    que NÃO seja um cabeçalho de conferência ou journal típico.
    Retorna None se não houver candidato confiável.
    """
    # Padrões de linhas que NÃO são títulos
    noise = re.compile(
        r'^(?:arXiv|doi|http|preprint|IEEE|ACM|Proceedings|\d{4}|©|\w{1,3}\s+\d)',
        re.IGNORECASE
    )
    lines = [l.strip() for l in text.split('\n')[:30] if l.strip()]
    for line in lines:
        if 10 < len(line) < 250 and not noise.match(line):
            return line
    return None


def parse_abstract(text: str) -> Optional[str]:
    """Retorna o abstract ou None — nunca texto inventado."""
    m = _RE_ABSTRACT.search(text)
    if m:
        return ' '.join(m.group(1).split())  # normaliza whitespace
    return None


def parse_keywords(text: str) -> list[str]:
    """Extrai keywords explicitamente declaradas. Retorna [] se ausente."""
    m = _RE_KEYWORDS.search(text)
    if not m:
        return []
    raw = m.group(1)
    # Separar por vírgula, ponto-e-vírgula ou hífen longo
    items = re.split(r'[,;]|\s{2,}', raw)
    return [k.strip() for k in items if k.strip() and len(k.strip()) > 2]


def parse_year(text: str) -> Optional[int]:
    """Procura ano nas primeiras 800 chars — zona mais densa de metadados."""
    m = _RE_YEAR.search(text[:800])
    return int(m.group(1)) if m else None


def parse_doi(text: str) -> Optional[str]:
    """Extrai DOI se presente. Retorna None caso contrário."""
    m = _RE_DOI.search(text[:1000])
    return m.group(1).rstrip('.,)') if m else None


def parse_sections(text: str) -> list[dict]:
    """
    Segmenta o documento em seções por heurística de headings.
    Conteúdo real do documento — nunca inventado.
    """
    lines = text.split('\n')
    sections, current_heading, buf, level = [], 'Preâmbulo', [], 0

    def _flush():
        if buf:
            sections.append({
                'heading': current_heading,
                'level': level,
                'content': ' '.join(buf).strip()
            })

    for line in lines:
        s = line.strip()
        if _RE_SECTION.match(s) and len(s) > 3:
            _flush()
            current_heading, buf = s, []
            # Determinar nível: numerado → nível pelo número; maiúsculas → 1
            num_match = re.match(r'^(\d+)', s)
            level = int(num_match.group(1)) if num_match else 1
        elif s:
            buf.append(s)

    _flush()
    return sections


def parse_references(text: str) -> list[dict]:
    """Extrai referências brutas. Nunca parseia além do texto bruto."""
    m = _RE_REF_SECTION.search(text)
    if not m:
        return []
    raw_refs = _RE_REF_SPLIT.split(m.group(1))
    return [
        {'raw_text': r.strip(),
         'parsed': {'authors': [], 'title': None, 'year': None, 'venue': None}}
        for r in raw_refs if len(r.strip()) > 20
    ][:100]  # cap para evitar JSONs gigantes


print('✅ Parsers de estrutura carregados.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 5 — MOTOR CENTRAL
#
# convert_pdf()   → converte 1 arquivo PDF em 1 JSON
# process_folder() → processa 1 pasta inteira (loop sobre convert_pdf)
# summarize()      → imprime e retorna estatísticas de uma lista de resultados
# run_all()        → ponto de entrada principal (orquestra tudo)
#
# DRY: toda lógica de conversão vive aqui. Células de execução
# apenas chamam run_all() ou process_folder().
# ═══════════════════════════════════════════════════════════════
import json
from datetime import datetime, timezone
from pathlib import Path

STATUS_ICON = {
    'converted':           '✅',
    'skipped_idempotent':  '⏩',
    'converted_warnings':  '⚠️',
    'failed':              '🔴',
}


def _build_document(pdf_path: Path, folder_name: str) -> dict:
    """
    Constrói o documento JSON a partir de um PDF.
    Campos ausentes → None/[]. Nunca inventa valores.
    """
    full_text, n_pages, tool, warnings = extract_text(pdf_path)

    has_text = bool(full_text.strip())

    return {
        'metadata': {
            'source_file':          pdf_path.name,
            'source_folder':        folder_name,
            'conversion_timestamp': datetime.now(timezone.utc).isoformat(),
            'converter_version':    CONVERTER_VERSION,
            'extraction_tool':      tool,
            'pages_total':          n_pages,
            'has_extractable_text': has_text,
            'extraction_warnings':  warnings,
        },
        'title':           parse_title(full_text)     if has_text else None,
        'authors':         [],      # complexo sem metadados externos
        'year':            parse_year(full_text)      if has_text else None,
        'venue':           None,    # não inferível com segurança
        'doi':             parse_doi(full_text)       if has_text else None,
        'abstract':        parse_abstract(full_text)  if has_text else None,
        'keywords':        parse_keywords(full_text)  if has_text else [],
        'sections':        parse_sections(full_text)  if has_text else [],
        'references':      parse_references(full_text)if has_text else [],
        'figures_detected': 0,      # requer análise de imagem
        'tables_detected':  0,
        'full_text_raw':    full_text,
    }


def convert_pdf(pdf_path: Path, output_folder: Path, folder_name: str) -> dict:
    """
    Converte 1 PDF em 1 JSON.
    Idempotente: retorna 'skipped_idempotent' se JSON já existe.
    Antifragil: qualquer exceção retorna status 'failed' com mensagem — nunca aborta.
    """
    output_path = output_folder / (pdf_path.stem + '.json')

    if output_path.exists():
        return {'status': 'skipped_idempotent', 'file': pdf_path.name, 'output': str(output_path), 'warnings': []}

    try:
        doc = _build_document(pdf_path, folder_name)

        # Serializar e validar antes de salvar (garante JSON válido)
        payload = json.dumps(doc, ensure_ascii=False, indent=2)
        json.loads(payload)  # validação: se falhar, não salva arquivo corrompido

        output_path.write_text(payload, encoding='utf-8')

        warnings = doc['metadata']['extraction_warnings']
        status = 'converted_warnings' if warnings else 'converted'
        return {'status': status, 'file': pdf_path.name, 'output': str(output_path), 'warnings': warnings}

    except Exception as e:
        return {'status': 'failed', 'file': pdf_path.name, 'output': None, 'warnings': [str(e)]}


def process_folder(folder_name: str, verbose: bool = True) -> list[dict]:
    """
    Processa todos os PDFs de uma pasta do catálogo.
    Única implementação de loop — reutilizada por todas as chamadas.
    """
    src  = FOLDER_CATALOG.get(folder_name)
    dest = OUTPUT_ROOT / folder_name

    if not src or not src.exists():
        print(f'❌ Pasta não encontrada: {folder_name}')
        return []

    pdfs = sorted(src.glob('*.pdf'))
    if not pdfs:
        print(f'⏹️  Nenhum PDF em {folder_name} — pasta pode já estar 100% convertida.')
        return []

    if verbose:
        print(f'\n📂 {folder_name}: {len(pdfs)} PDFs encontrados')
        print('─' * 64)

    results = []
    for pdf in pdfs:
        r = convert_pdf(pdf, dest, folder_name)
        results.append(r)
        if verbose:
            icon = STATUS_ICON.get(r['status'], '❓')
            print(f'{icon} {r["file"]}')
            for w in r.get('warnings', []):
                print(f'   ↳ {w}')

    return results


def summarize(results: list[dict], label: str = '') -> dict:
    """Calcula e imprime estatísticas de uma lista de resultados."""
    counts = {s: sum(1 for r in results if r['status'] == s) for s in STATUS_ICON}
    if label:
        print(f'\n📊 {label}')
    for status, icon in STATUS_ICON.items():
        print(f'   {icon} {status:<22} {counts[status]}')
    return counts


def save_report(all_results: list[dict], folders_processed: list[str]) -> Path:
    """Salva relatório JSON consolidado em articles-outputs/."""
    # Detectar PDFs ainda não convertidos em TODAS as pastas conhecidas
    remaining = [
        {'folder': fn, 'file': p.name}
        for fn, fp in FOLDER_CATALOG.items()
        if fp.exists()
        for p in fp.glob('*.pdf')
        if not (OUTPUT_ROOT / fn / (p.stem + '.json')).exists()
    ]

    report = {
        'timestamp':               datetime.now(timezone.utc).isoformat(),
        'converter_version':       CONVERTER_VERSION,
        'folders_processed':       folders_processed,
        'total_pdfs_found':        len(all_results),
        'total_converted':         sum(1 for r in all_results if r['status'] == 'converted'),
        'total_converted_warnings':sum(1 for r in all_results if r['status'] == 'converted_warnings'),
        'total_skipped_idempotent':sum(1 for r in all_results if r['status'] == 'skipped_idempotent'),
        'total_failed':            sum(1 for r in all_results if r['status'] == 'failed'),
        'remaining_unconverted':   len(remaining),
        'remaining_files':         remaining,
        'output_path':             str(OUTPUT_ROOT),
        'details':                 all_results,
    }

    ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    path = OUTPUT_ROOT / f'conversion_report_{ts}.json'
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    return path


def run_all(extra_folders: list[str] = None) -> list[dict]:
    """
    Ponto de entrada principal.
    Sempre processa 'best-sources', depois as pastas em extra_folders.
    Gera relatório consolidado ao final.
    """
    folders = ['best-sources'] + (extra_folders or [])
    all_results = []

    for folder in folders:
        results = process_folder(folder)
        all_results.extend(results)

    print('\n' + '=' * 64)
    summarize(all_results, label='RESUMO GLOBAL')

    report_path = save_report(all_results, folders)
    print(f'\n📝 Relatório salvo em:\n   {report_path}')

    return all_results


print('✅ Motor ATLAS v2 pronto.')
print('   → Para executar: run_all() ou process_folder("best-sources")')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6 — EXECUÇÃO INTERATIVA
#
# OPÇÃO A (recomendado): run_all() com seleção de pastas extras
# OPÇÃO B: Roda apenas best-sources automaticamente
# OPÇÃO C: Linha de comando / scripts externos chamam run_all()
# ═══════════════════════════════════════════════════════════════

# ── Verificar critério de parada antes de rodar ────────────────
def _count_pending(folder_name: str) -> int:
    src = FOLDER_CATALOG.get(folder_name)
    if not src or not src.exists():
        return 0
    pdfs = list(src.glob('*.pdf'))
    converted = [(OUTPUT_ROOT / folder_name / (p.stem + '.json')).exists() for p in pdfs]
    return sum(1 for c in converted if not c)


pending_best = _count_pending('best-sources')

if pending_best == 0:
    print('⏹️  best-sources: todos os PDFs já foram convertidos.')
    print('   Critério de parada atingido para esta pasta.')
else:
    print(f'📋 {pending_best} PDFs pendentes em best-sources.')

# ── Seleção de pastas adicionais ───────────────────────────────
print()
print('═' * 64)
print('Deseja converter pastas adicionais após best-sources?')
for i, name in enumerate(list(FOLDER_CATALOG.keys())[1:], 1):
    pending = _count_pending(name)
    print(f'  [{i}] {name} ({pending} pendentes)')
print(f'  [{len(FOLDER_CATALOG)}] Todas as pastas')
print(f'  [0] Apenas best-sources')
print('═' * 64)

raw = input('Escolha [0 / 1 / 2 / 3]: ').strip()
optional_keys = list(FOLDER_CATALOG.keys())[1:]  # exclui best-sources

extras = []
if raw == '1':
    extras = [optional_keys[0]]
elif raw == '2':
    extras = [optional_keys[1]]
elif raw == '3':
    extras = optional_keys
# raw == '0' → extras = [] → apenas best-sources

print(f'\n▶️  Iniciando conversão... pastas: {["best-sources"] + extras}')
print()

results = run_all(extra_folders=extras)